# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 30 |
| **Integrantes** | Andres Felipe Grisales Vahos · Hanna Jineth Contreras Salinas · Arley Steven Franco Galeano |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | domingo 30 de agosto |



---
## 1. Contexto y problema

Wanderbricks opera un marketplace de alquiler vacacional donde los usuarios navegan propiedades a través de distintos dispositivos y canales antes de decidir reservar. Sin embargo, no toda esa actividad de navegación se traduce en una reserva confirmada y pagada: hay tráfico que se pierde en el camino, y hoy esa información vive dispersa entre el registro de navegación, las reservas y los pagos, sin un modelo que permita cruzarlos.

El equipo de marketing necesita entender qué combinaciones de dispositivo y canal de origen (referrer) están asociadas a reservas que efectivamente se confirman y pagan, frente a las que solo generan tráfico sin conversión, información clave para decidir en qué canales invertir. Adicionalmente, interesa saber si esas reservas "bien convertidas" también se traducen en mejores reseñas, lo que indicaría que el canal de origen no solo predice conversión, sino también satisfacción.

Esta base de datos analítica debe responder preguntas de negocio como: (1) ¿qué dispositivos y canales de origen presentan mayor tasa de conversión de navegación a reserva pagada?, (2) ¿existen patrones de abandono, navegación sin reserva concentrados en ciertos canales?, y (3) ¿las reservas provenientes de los canales con mejor conversión también obtienen mejores calificaciones en las reseñas?

---
## 2. Descripción de los datos

Antes de diseñar cualquier consulta necesitábamos saber con qué estábamos trabajando, saber cuántos registros tiene cada tabla, qué columnas maneja y de qué tipo es cada una. Por eso, en la siguiente celda recorremos las 4 tablas que elegimos (bookings, clickstream, reviews, payments) y les imprimimos el esquema completo y el número de filas. Esto nos ayuda porque, antes de escribir una sola consulta, ya sabemos qué campos existen, cómo se llaman exactamente y qué tan grande es cada tabla así evitamos suposiciones erróneas más adelante, sobre todo con clickstream, que como se ve abajo tiene una estructura distinta a las otras tres.

In [0]:
# Exploración de las tablas seleccionadas por el grupo
tablas_clave = ["bookings", "clickstream", "reviews", "payments"]

for tabla in tablas_clave:
    print(f"\n{'='*50}")
    print(f"TABLA: {tabla}")
    print(f"{'='*50}")
    df = spark.table(f"samples.wanderbricks.{tabla}")
    print(f"Filas: {df.count()}")
    df.printSchema()

### Volumen y variedad:

Trabajamos con 4 de las 16 tablas del dataset Wanderbricks: bookings (72,247 filas), clickstream (100,000 filas), reviews (99,793 filas) y payments (49,638 filas). El volumen es consistente con un marketplace de tamaño mediano, es decir, suficiente para observar patrones de comportamiento sin ser un dataset masivo.

### Tipos de datos y estructuras:

bookings y payments siguen un esquema tabular plano estándar (long, string, date, timestamp, float/decimal). clickstream, en cambio, tiene un campo metadata que es un struct anidado con device y referrer adentro, esta es la tabla que vamos a usar para el requisito de datos semiestructurados, y además conecta directo con nuestra pregunta de negocio, porque justo queremos saber qué dispositivo y canal de origen generan más conversión. reviews tiene comment, que es texto libre.

### Calidad observada:

En reviews encontramos un campo is_deleted, lo que significa que las reseñas eliminadas no se borran realmente, solo quedan marcadas. Esto nos toca tenerlo en cuenta en las consultas, porque si no filtramos por is_deleted = false, vamos a contar reseñas que en la práctica ya no deberían aparecer.

### Relaciones entre tablas:

bookings es la tabla que conecta a todas las demás. Cada pago en payments está ligado a una reserva específica a través de booking_id, así que una misma reserva puede tener uno o varios pagos asociados. reviews también se conecta con bookings, pero de una forma más completa, ya que comparte booking_id, user_id y property_id al mismo tiempo, porque una reseña habla tanto de la reserva como del usuario que la hizo y de la propiedad que calificó. Con clickstream la relación es un poco distinta, es decir no hay una llave directa hacia bookings, pero comparte user_id y property_id, lo que nos permite reconstruir qué navegación tuvo un usuario antes de terminar reservando (o sin reservar). Esta cadena de relaciones (navegación, reserva, pago y reseña) es justo lo que necesitamos para responder nuestra pregunta de negocio.

#### Periodo cubierto por los datos

Además de conocer el volumen y los tipos de datos, nos pareció importante revisar en qué rango de fechas se mueve cada tabla. Esto nos sirve para saber si estamos analizando un periodo corto o largo, y para detectar a tiempo si hay algo raro en las fechas que debamos tener en cuenta antes de sacar conclusiones más adelante.

In [0]:
%sql
SELECT
    (SELECT MIN(timestamp) FROM bigdata_grupo30.wanderbricks.clickstream_plata) AS min_clickstream,
    (SELECT MAX(timestamp) FROM bigdata_grupo30.wanderbricks.clickstream_plata) AS max_clickstream,
    (SELECT MIN(created_at) FROM bigdata_grupo30.wanderbricks.bookings_plata) AS min_bookings,
    (SELECT MAX(created_at) FROM bigdata_grupo30.wanderbricks.bookings_plata) AS max_bookings

**Interpretación**

Con esta consulta buscamos la fecha más antigua y la más reciente registradas en `clickstream_plata` y en `bookings_plata`, para saber con exactitud qué ventana de tiempo cubren nuestros datos. Esto nos ayuda porque cualquier conclusión que saquemos en las secciones 5 y 6 solo es válida para ese periodo.

El resultado nos mostró algo que no esperábamos: `bookings_plata` sí cubre un rango amplio, del 8 de diciembre de 2022 al 30 de julio de 2025, pero `clickstream_plata` solo cubre unos segundos del 4 de septiembre de 2025 (de 23:26:48 a 23:27:01). Es decir, todos los eventos de navegación quedaron registrados casi al mismo instante, no distribuidos naturalmente en el tiempo como las reservas. Esto es una limitación importante porque no estamos analizando un histórico real de comportamiento de navegación a lo largo de meses, sino una muestra generada de una sola vez.

---
## 3. Decisiones de diseño y justificación

Para decidir qué paradigma de base de datos usar, comparamos tres opciones: relacional, NoSQL y lakehouse, frente a cuatro necesidades concretas que encontramos mientras explorábamos los datos de Wanderbricks en las secciones 2 y 4.

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| **Struct anidado en `clickstream`** (campo `metadata` con `device` y `referrer` adentro, ver 4.4) | Requiere aplanar el struct antes de cargarlo, porque el modelo relacional clásico no maneja bien columnas anidadas; forzaría separar esa información en tablas o columnas desde el origen. | Maneja structs y documentos anidados de forma nativa es su fortaleza, pero no ofrece SQL estándar ni joins eficientes con las otras 3 tablas tabulares del caso. | Puede guardar el struct tal cual llega en la capa bronce (como hicimos) y aplanarlo solo cuando conviene en la capa plata, sin perder el dato crudo original. | **Lakehouse**: nos deja quedarnos con el dato crudo *y* trabajarlo tabular cuando lo necesitamos, sin elegir un solo formato desde el inicio. |
| **Transacciones ACID sobre `bookings`/`payments`** (UPDATE del 19% de IVA y DELETE de montos nulos, ver 4.5) | Ofrece ACID de forma nativa y madura es su punto más fuerte. | La mayoría de motores NoSQL solo dan consistencia eventual, o ACID limitado a un solo documento, no a través de varias tablas. | Delta Lake (la capa que usamos) da garantías ACID equivalentes a un motor relacional sobre archivos de un data lake, como demostramos en 4.5. | **Lakehouse**: iguala la garantía transaccional del modelo relacional sin sacrificar el resto de ventajas de un lake. |
| **Auditoría y recuperación de versiones anteriores** (time travel, ver 4.5) | No existe de forma nativa; hay que construirla a mano con tablas de historial o triggers. | Tampoco es una característica estándar del paradigma. | Es una función incorporada (`DESCRIBE HISTORY`, `VERSION AS OF`), que usamos para recuperar el monto total antes del ajuste del 19%. | **Lakehouse**: nos ahorra construir nuestra propia solución de auditoría desde cero. |
| **Volumen y variedad combinados** (4 tablas, hasta 100,000 filas, mezcla de datos tabulares y semiestructurados) | Cómodo para el volumen actual, pero escala verticalmente y no está pensado para mezclar estructuras distintas en el mismo motor. | Escala horizontalmente muy bien, pero perderíamos la capacidad de hacer joins SQL directos entre `bookings`, `payments`, `reviews` y `clickstream`, que es justo lo que necesitan nuestras 5 consultas. | Escala horizontalmente igual que un data lake y permite SQL completo entre todas las tablas, nazcan estructuradas o no. | **Lakehouse**: es la única opción que no nos obliga a sacrificar volumen o variedad para conservar la capacidad de consulta SQL que necesitamos. |

**Conclusión de la decisión:** elegimos lakehouse (Delta Lake sobre Databricks) porque fue la única opción que resolvió las cuatro necesidades a la vez que son: dato semiestructurado, transacciones confiables, auditoría y volumen, mientras que relacional y NoSQL solo cubrían una parte cada uno.

**Referencias (APA 7):**

Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). Lakehouse: A new generation of open source systems and data architecture. *Proceedings of CIDR 2021*.

Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM, 13*(6), 377–387.

Sadalage, P. J., & Fowler, M. (2012). *NoSQL distilled: A brief guide to the emerging world of polyglot persistence*. Addison-Wesley.

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

En esta sección construimos, paso a paso, la base de datos analítica: primero creamos nuestro propio espacio de trabajo (catálogo, esquema y volumen), después la capa bronce con los datos crudos, luego la capa plata ya limpia, y por último demostramos con código las garantías que ofrece Delta Lake. Cada subsección tiene su código seguido de la interpretación de lo que hicimos y por qué.

Antes de crear cualquier tabla necesitábamos un espacio propio donde trabajar, separado del catálogo de solo lectura `samples.wanderbricks`  ahí no tenemos permiso de escritura, y tampoco deberíamos tocar los datos originales del curso. Por eso creamos nuestro propio catálogo (`bigdata_grupo30`), con su esquema y su volumen, siguiendo la misma jerarquía catálogo, esquema, tabla que ya conocíamos de bases de datos relacionales.

In [0]:
GRUPO = "30"
CATALOGO = f"bigdata_grupo{GRUPO}"
ESQUEMA = "wanderbricks"
VOLUMEN = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

print(f"✅ Catálogo activo: {CATALOGO}")
print(f"✅ Esquema activo:  {ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

La capa bronce guarda una copia fiel de los datos originales, sin transformar nada, solo agregando dos columnas de auditoría, cuándo se ingirió el dato (`_ingesta_ts`) y de dónde vino (`_origen`). La idea es tener siempre un respaldo del dato tal como llegó, por si más adelante necesitamos revisar algo o corregir una regla de limpieza que quedó mal calibrada porque no podemos arreglar lo que ya borramos.

In [0]:
from pyspark.sql import functions as F

tablas_negocio = ["bookings", "clickstream", "reviews", "payments"]

for tabla in tablas_negocio:
    tabla_bronce = f"{CATALOGO}.{ESQUEMA}.{tabla}_bronce"
    
    df = (spark.table(f"samples.wanderbricks.{tabla}")
            .withColumn("_ingesta_ts", F.current_timestamp())
            .withColumn("_origen", F.lit(f"samples.wanderbricks.{tabla}")))
    
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabla_bronce))
    
    print(f"✅ {tabla_bronce}: {df.count():,} registros, {len(df.columns)} columnas")

**Interpretación**

Construimos la capa bronce para las 4 tablas del caso de negocio (bookings, clickstream, reviews, payments), cada una con sus dos columnas de auditoría (_ingesta_ts y _origen). Usamos un mismo bloque de código reutilizable en lugar de repetirlo 4 veces, porque el proceso de ingesta es idéntico para todas ya que solo cambia la tabla de origen. Esto también facilita que, si mañana se agrega una quinta tabla al análisis, solo haya que sumarla a la lista.

### 4.3 Capa plata — datos limpios y tipados

En la capa plata ya no trabajamos con el dato crudo, sino con una versión limpia y confiable: quitamos duplicados por la llave principal de cada tabla, descartamos registros que no sirven para el análisis (como montos nulos), y en `reviews` excluimos las reseñas marcadas como borradas. Esta es la capa que realmente usamos en las consultas de la Sección 5.

#### Limpieza y transformación de las 4 tablas

In [0]:
from pyspark.sql import functions as F

# bookings_plata: sin duplicados, sin montos nulos, tipo explícito
bookings_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_bronce")
    .dropDuplicates(["booking_id"])
    .filter(F.col("total_amount").isNotNull())
    .withColumn("total_amount", F.col("total_amount").cast("decimal(10,2)")))
bookings_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bookings_plata")

# payments_plata: sin duplicados, sin montos nulos
payments_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.payments_bronce")
    .dropDuplicates(["payment_id"])
    .filter(F.col("amount").isNotNull()))
payments_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.payments_plata")

# reviews_plata: quitamos las borradas lógicamente (is_deleted)
reviews_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.reviews_bronce")
    .filter(F.col("is_deleted") == False)
    .dropDuplicates(["review_id"]))
reviews_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.reviews_plata")

# clickstream_plata: "desempacamos" el struct metadata en columnas normales
clickstream_plata = (spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_bronce")
    .withColumn("device", F.col("metadata.device"))
    .withColumn("referrer", F.col("metadata.referrer"))
    .drop("metadata"))
clickstream_plata.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")

print("✅ Capa plata creada: bookings, payments, reviews, clickstream")

**Interpretación**

Tomamos cada tabla de bronce y le aplicamos limpieza específica según lo que necesitaba. En bookings y payments quitamos duplicados por su llave principal (booking_id y payment_id) y descartamos los registros con monto nulo, porque no aportan nada al análisis financiero. En reviews excluimos las reseñas marcadas como borradas (is_deleted = true), ya que aunque siguen guardadas en bronce por auditoría, no deberían contarse como reseñas activas. En clickstream, sacamos device y referrer de adentro del struct metadata y los dejamos como columnas normales, para poder agruparlos y filtrarlos directamente en las consultas sin tener que navegar la estructura anidada cada vez.

### 4.4 Datos semiestructurados


La rúbrica nos pide demostrar que sabemos manejar datos anidados, y `clickstream` nos da el ejemplo perfecto, trae un campo llamado `metadata` que es un struct con dos datos adentro (`device` y `referrer`). Acá mostramos cómo se ve esa estructura anidada por dentro, y cómo la convertimos en columnas normales para poder agruparla y cruzarla con otras tablas sin tener que navegar el struct cada vez.

#### Aplanando el struct anidado de clickstream

In [0]:
# Mostramos el struct anidado ANTES de aplanarlo
print("Estructura original de clickstream (con struct anidado):")
spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_bronce").printSchema()

# Y el resultado después de aplanarlo (ya en la capa plata)
print("\nDespués de aplanar el struct 'metadata':")
spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_plata").select("device", "referrer").show(5)

**Interpretación**

clickstream trae el campo metadata como un struct anidado que contiene device y referrer. En SQL o PySpark, acceder a un valor dentro de un struct requiere notación de punto (metadata.device), lo que hace las consultas más incómodas de escribir y de leer. Por eso, en la capa plata extrajimos esos dos campos y los dejamos como columnas de primer nivel de manera que así se pueden agrupar, filtrar y unir con otras tablas de forma directa, igual que cualquier columna normal. Esta es la estructura semiestructurada que trabajamos para cumplir el requisito de datos anidados de la evidencia.

### 4.5 Propiedades del lakehouse


En esta parte demostramos, con código y no solo en teoría, las 4 garantías que ofrece Delta Lake y es que las transacciones sean atómicas y consistentes (ACID), que quede un historial completo de todo lo que le pasa a una tabla, que podamos consultar versiones anteriores sin haber guardado copias manuales (time travel), y que el esquema pueda evolucionar sin romper nada. Estas cuatro cosas son justamente lo que un modelo relacional o NoSQL tradicional no ofrece de fábrica, y son la base de lo que argumentamos en la Sección 3.

#### ACID — Atomicidad y consistencia:

Para demostrar esto, modificamos datos reales: subimos el monto de las reservas confirmadas un 19%, y después intentamos borrar los registros con monto nulo. Lo que nos interesa observar es qué pasa con el conteo de filas antes y después de cada operación, y por qué.

In [0]:
TABLA_BRONCE = f"{CATALOGO}.{ESQUEMA}.bookings_bronce"

# Capturamos la versión actual ANTES de modificar nada, en vez de fijar un número
version_base = spark.sql(f"DESCRIBE HISTORY {TABLA_BRONCE} LIMIT 1").select("version").first()["version"]
print(f"Versión base: {version_base}")

antes = spark.table(TABLA_BRONCE).count()

spark.sql(f"""
    UPDATE {TABLA_BRONCE}
    SET total_amount = total_amount * 1.19
    WHERE status = 'confirmed'
""")

spark.sql(f"DELETE FROM {TABLA_BRONCE} WHERE total_amount IS NULL")

despues = spark.table(TABLA_BRONCE).count()

print(f"Registros antes:   {antes:,}")
print(f"Registros después: {despues:,}")
print(f"Eliminados:        {antes - despues:,}")

In [0]:
confirmadas = spark.table(TABLA_BRONCE).filter("status = 'confirmed'").count()
print(f"Reservas con status 'confirmed': {confirmadas:,}")

**Interpretación**

Actualizamos total_amount en las reservas confirmadas, subiéndolo un 19%, y después intentamos borrar los registros con total_amount nulo. El conteo se mantiene igual antes y después (72,247), porque el UPDATE no agrega ni quita filas solo cambia el valor de la columna en las 17,948 reservas que tenían status = 'confirmed'. El DELETE tampoco eliminó nada, y eso en realidad nos dice algo útil, no hay ningún registro con total_amount vacío en la tabla, así que no tenemos ese problema de calidad en los datos de reservas.

#### Historial de transacciones:

Después de modificar la tabla, Delta guarda un registro de cada operación que le hicimos, incluso las que no cambiaron nada (como el DELETE que no encontró nulos). Acá lo consultamos para ver el historial completo de la tabla.

In [0]:
display(spark.sql(f"""
    DESCRIBE HISTORY {TABLA_BRONCE}
""").select("version", "timestamp", "operation", "operationMetrics"))

**Interpretación**

El historial muestra 9 versiones en total. Las primeras cinco (0 a 4) corresponden a cuando construimos por primera vez esta tabla en las pruebas iniciales del laboratorio. La versión 5 es la que reconstruye la capa bronce para las 4 tablas del proyecto, usando overwrite, lo que reinició la tabla a su estado original de 72,247 filas. Las versiones 6, 7 y 8 corresponden a las operaciones de esta sección: el UPDATE sobre las reservas confirmadas, el DELETE que no encontró nulos, y una operación OPTIMIZE que Delta ejecutó automáticamente para compactar archivos pequeños, sin que nosotros la pidiéramos. Esto demuestra que Delta lleva un registro completo de todo lo que le sucede a una tabla a lo largo del tiempo, no solo de la sesión actual.

#### Viaje en el tiempo (time travel):


Con el historial ya identificado, demostramos que podemos consultar cómo estaban los datos exactamente antes de nuestros cambios, sin haber guardado ninguna copia manual de respaldo, comparamos el monto total de una versión anterior contra el actual.

In [0]:
display(spark.sql(f"""
    SELECT 'versión base' AS version, ROUND(SUM(total_amount), 0) AS monto
    FROM {TABLA_BRONCE} VERSION AS OF {version_base}
    UNION ALL
    SELECT 'versión actual' AS version, ROUND(SUM(total_amount), 0) AS monto
    FROM {TABLA_BRONCE}
"""))

In [0]:
resultado = spark.sql(f"""
    SELECT
        (SELECT ROUND(SUM(total_amount), 0) FROM {TABLA_BRONCE} VERSION AS OF {version_base}) AS monto_base,
        (SELECT ROUND(SUM(total_amount), 0) FROM {TABLA_BRONCE}) AS monto_actual
""").first()

diferencia = resultado["monto_actual"] - resultado["monto_base"]
print(f"Diferencia: ${diferencia:,.0f}")

**Interpretación**

Comparamos el monto total de total_amount entre la versión 5 (el estado de la tabla justo antes de las operaciones de esta sección) y la versión actual. La diferencia de $1,883,650 corresponde exactamente al ajuste del 19% que aplicamos con el UPDATE. Esto demuestra que, sin haber guardado ninguna copia manual, podemos reconstruir el estado exacto de los datos en cualquier punto anterior, si el ajuste hubiera sido un error, podríamos volver a la versión 5 sin perder nada.

#### Evolución de esquema:

Por último, simulamos que llega un dato nuevo que la tabla original no tenía (el canal de origen de la reserva), para comprobar si Delta lo acepta sin romper la tabla ni perder los datos que ya existían.

In [0]:
# Simulamos que llega un nuevo campo: canal de origen de la reserva
from pyspark.sql import Row

nuevas_filas = spark.createDataFrame([
    Row(booking_id=999901, user_id=10001, property_id=5001, canal="instagram"),
    Row(booking_id=999902, user_id=10002, property_id=5002, canal="facebook"),
    Row(booking_id=999903, user_id=10003, property_id=5003, canal="google_ads")
])

(nuevas_filas.write
   .format("delta")
   .mode("append")
   .option("mergeSchema", "true")
   .saveAsTable(TABLA_BRONCE))

print("Columnas después de la evolución:")
for col in spark.table(TABLA_BRONCE).columns:
    print(f"  · {col}")

In [0]:
display(spark.sql(f"""
    SELECT canal, COUNT(*) AS registros
    FROM {TABLA_BRONCE}
    GROUP BY canal
"""))

**Interpretación**

Simulamos la llegada de un dato nuevo que la tabla original no contemplaba, el canal de origen de la reserva. Al escribir con mergeSchema=true, Delta aceptó la columna nueva sin rechazar el resto de los datos ni romper la tabla, extendiendo el esquema de 12 a 13 columnas automáticamente. Las 72,247 filas originales quedan con canal en nulo, porque nunca tuvieron ese dato solo las 3 filas nuevas lo tienen. Esto demuestra que el esquema puede evolucionar sobre la marcha, algo especialmente útil cuando el negocio empieza a capturar información que antes no registraba.

---
## 5. Resultados


### 5.1 Tasa de conversión por dispositivo y canal

*¿qué dispositivos/canales convierten mejor?*

**SQL:**

In [0]:
%sql
WITH navegantes AS (
    SELECT DISTINCT user_id, device, referrer
    FROM bigdata_grupo30.wanderbricks.clickstream_plata
),
convertidos AS (
    SELECT DISTINCT b.user_id
    FROM bigdata_grupo30.wanderbricks.bookings_plata b
    JOIN bigdata_grupo30.wanderbricks.payments_plata p ON b.booking_id = p.booking_id
    WHERE b.status = 'confirmed' AND p.status = 'completed'
)
SELECT n.device, n.referrer,
       COUNT(DISTINCT n.user_id) AS usuarios_navegaron,
       COUNT(DISTINCT c.user_id) AS usuarios_convirtieron,
       ROUND(COUNT(DISTINCT c.user_id) / COUNT(DISTINCT n.user_id) * 100, 2) AS tasa_conversion_pct
FROM navegantes n
LEFT JOIN convertidos c ON n.user_id = c.user_id
GROUP BY n.device, n.referrer
ORDER BY tasa_conversion_pct DESC

**PySpark (mismo resultado, otra forma de escribirlo):**

In [0]:
from pyspark.sql import functions as F
navegantes = (spark.table("bigdata_grupo30.wanderbricks.clickstream_plata")
    .select("user_id", "device", "referrer").distinct())

convertidos = (spark.table("bigdata_grupo30.wanderbricks.bookings_plata").alias("b")
    .join(spark.table("bigdata_grupo30.wanderbricks.payments_plata").alias("p"), "booking_id")
    .filter("b.status = 'confirmed' AND p.status = 'completed'")
    .select("b.user_id").distinct())

resultado = (navegantes.join(convertidos.withColumnRenamed("user_id", "user_id_conv"),
        navegantes.user_id == F.col("user_id_conv"), "left")
    .groupBy("device", "referrer")
    .agg(F.countDistinct("user_id").alias("usuarios_navegaron"),
         F.countDistinct("user_id_conv").alias("usuarios_convirtieron"))
    .withColumn("tasa_conversion_pct", F.round(F.col("usuarios_convirtieron") / F.col("usuarios_navegaron") * 100, 2))
    .orderBy(F.desc("tasa_conversion_pct")))
resultado.display()

**Interpretación**

Qué combinaciones de dispositivo/canal convierten mejor. No basta con contar cuántos navegan por canal si no  el dato útil es qué porcentaje de los que navegan realmente terminan pagando. Por eso se construyó como una tasa (convertidos ÷ navegantes), no como un conteo simple.

### 5.2 Patrones de abandono por canal
*¿dónde se concentra el tráfico que no reserva?*

**SQL:**

In [0]:
%sql
SELECT c.device,c.referrer ,
       COUNT(DISTINCT c.user_id) AS usuarios_sin_reserva
FROM bigdata_grupo30.wanderbricks.clickstream_plata c
LEFT JOIN bigdata_grupo30.wanderbricks.bookings_plata b ON c.user_id = b.user_id
WHERE b.user_id IS NULL
GROUP BY c.referrer, c.device
ORDER BY usuarios_sin_reserva DESC

**PySpark (mismo resultado, otra forma de escribirlo):**

In [0]:
from pyspark.sql import functions as F
resultado = (spark.table("bigdata_grupo30.wanderbricks.clickstream_plata").alias("c")
    .join(spark.table("bigdata_grupo30.wanderbricks.bookings_plata").alias("b"), "user_id", "left")
    .filter(F.col("b.user_id").isNull())
    .groupBy("c.referrer", "c.device")
    .agg(F.countDistinct("c.user_id").alias("usuarios_sin_reserva"))
    .orderBy(F.desc("usuarios_sin_reserva")))
resultado.display()

**Interpretación**

Complementa la consulta 1 desde el ángulo opuesto: en vez de medir quién convierte, mide dónde se concentra el tráfico que no convierte. Es la misma pregunta de negocio (2) vista al revés, y es necesaria porque un canal puede traer mucho volumen sin que eso signifique que convierte bien.


### 5.3 Rating promedio por canal de origen
*¿los canales que más convierten también dejan mejores reseñas?*

**SQL:**

In [0]:
%sql
WITH canal_por_propiedad AS (
    SELECT property_id, referrer,
           COUNT(*) AS visitas,
           ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY COUNT(*) DESC) AS rn
    FROM bigdata_grupo30.wanderbricks.clickstream_plata
    GROUP BY property_id, referrer
)
SELECT cp.referrer AS canal_predominante,
       ROUND(AVG(r.rating), 2) AS rating_promedio,
       COUNT(DISTINCT r.property_id) AS num_propiedades,
       COUNT(r.review_id) AS num_resenas
FROM bigdata_grupo30.wanderbricks.reviews_plata r
JOIN canal_por_propiedad cp ON r.property_id = cp.property_id AND cp.rn = 1
GROUP BY cp.referrer
ORDER BY rating_promedio DESC

**PySpark (mismo resultado, otra forma de escribirlo):**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Canal predominante por propiedad (el que trae más visitas a esa propiedad)
w = Window.partitionBy("property_id").orderBy(F.desc("visitas"))

canal_por_propiedad = (spark.table("bigdata_grupo30.wanderbricks.clickstream_plata")
    .groupBy("property_id", "referrer")
    .agg(F.count("*").alias("visitas"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("property_id", F.col("referrer").alias("canal_predominante")))

resultado = (spark.table("bigdata_grupo30.wanderbricks.reviews_plata").alias("r")
    .join(canal_por_propiedad, "property_id")
    .groupBy("canal_predominante")
    .agg(
        F.round(F.avg("rating"), 2).alias("rating_promedio"),
        F.countDistinct("property_id").alias("num_propiedades"),
        F.count("review_id").alias("num_resenas"))
    .orderBy(F.desc("rating_promedio")))

resultado.display()

**Interpretación**

Aquí se tomó una decisión de diseño explícita: el cruce original (mismo usuario + misma propiedad entre clickstream y reviews) daba 0 resultados, porque se confirmó que ambas tablas no comparten población real de usuarios. Se cambió el criterio de atribución a canal predominante por propiedad (el referrer que más tráfico le trae a cada propiedad), lo cual sí es calculable porque ambas tablas comparten el mismo universo de property_id. Esta consulta responde la pregunta 3: si el canal de origen se relaciona con la calidad percibida.



### 5.4 Ingresos por canal
*No solo quién convierte más, sino quién genera más ingresos*

**SQL:**

In [0]:
%sql
WITH canal_por_propiedad AS (
    SELECT property_id, referrer,
           COUNT(*) AS visitas,
           ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY COUNT(*) DESC) AS rn
    FROM bigdata_grupo30.wanderbricks.clickstream_plata
    GROUP BY property_id, referrer
)
SELECT cp.referrer AS canal_predominante,
       SUM(CASE WHEN p.status = 'completed' THEN p.amount ELSE 0 END) AS ingresos_netos,
       SUM(CASE WHEN p.status = 'refunded' THEN p.amount ELSE 0 END) AS monto_reembolsado,
       COUNT(DISTINCT CASE WHEN p.status = 'completed' THEN p.booking_id END) AS reservas_pagadas,
       COUNT(DISTINCT CASE WHEN p.status = 'refunded' THEN p.booking_id END) AS reservas_reembolsadas,
       ROUND(
           SUM(CASE WHEN p.status = 'refunded' THEN p.amount ELSE 0 END) /
           NULLIF(SUM(CASE WHEN p.status IN ('completed','refunded') THEN p.amount ELSE 0 END), 0) * 100, 2
       ) AS tasa_reembolso_pct
FROM bigdata_grupo30.wanderbricks.payments_plata p
JOIN bigdata_grupo30.wanderbricks.bookings_plata b ON p.booking_id = b.booking_id
JOIN canal_por_propiedad cp ON b.property_id = cp.property_id AND cp.rn = 1
GROUP BY cp.referrer
ORDER BY ingresos_netos DESC

**PySpark (mismo resultado, otra forma de escribirlo):**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

w = Window.partitionBy("property_id").orderBy(F.desc("visitas"))

canal_por_propiedad = (spark.table("bigdata_grupo30.wanderbricks.clickstream_plata")
    .groupBy("property_id", "referrer")
    .agg(F.count("*").alias("visitas"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("property_id", F.col("referrer").alias("canal_predominante")))

df_pagos = spark.table("bigdata_grupo30.wanderbricks.payments_plata").alias("p")
df_bookings = spark.table("bigdata_grupo30.wanderbricks.bookings_plata").alias("b")

pagos = (df_pagos.join(df_bookings, "booking_id")
    .join(canal_por_propiedad, F.col("b.property_id") == canal_por_propiedad.property_id))

resultado = (pagos.groupBy("canal_predominante")
    .agg(
        F.sum(F.when(F.col("p.status") == "completed", F.col("p.amount")).otherwise(0)).alias("ingresos_netos"),
        F.sum(F.when(F.col("p.status") == "refunded", F.col("p.amount")).otherwise(0)).alias("monto_reembolsado"),
        F.countDistinct(F.when(F.col("p.status") == "completed", F.col("booking_id"))).alias("reservas_pagadas"),
        F.countDistinct(F.when(F.col("p.status") == "refunded", F.col("booking_id"))).alias("reservas_reembolsadas"))
    .withColumn("tasa_reembolso_pct",
        F.round(F.col("monto_reembolsado") / (F.col("ingresos_netos") + F.col("monto_reembolsado")) * 100, 2))
    .orderBy(F.desc("ingresos_netos")))

resultado.display()

**Interpretación**

la consulta 1 (conversión) por sí sola no dice qué canal es más valioso en términos de dinero real — un canal puede convertir muchos usuarios pero con tickets pequeños o con muchos reembolsos. Se usó la misma lógica de canal predominante por propiedad que en la consulta 3, y se separó completed de refunded para no inflar el ingreso con dinero que la empresa terminó devolviendo.


### 5.5 Tasa de cancelación por canal
*¿algunos canales cancelan más?*

**SQL:**

In [0]:
%sql
WITH canal_por_propiedad AS (
    SELECT property_id, referrer,
           ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY COUNT(*) DESC) AS rn
    FROM bigdata_grupo30.wanderbricks.clickstream_plata
    GROUP BY property_id, referrer
)
SELECT cp.referrer AS canal_predominante,
       COUNT(*) AS total_reservas,
       COUNT(CASE WHEN b.status = 'confirmed' THEN 1 END) AS confirmadas,
       COUNT(CASE WHEN b.status = 'cancelled' THEN 1 END) AS canceladas,
       ROUND(COUNT(CASE WHEN b.status = 'cancelled' THEN 1 END) / COUNT(*) * 100, 2) AS tasa_cancelacion_pct
FROM bigdata_grupo30.wanderbricks.bookings_plata b
JOIN canal_por_propiedad cp ON b.property_id = cp.property_id AND cp.rn = 1
GROUP BY cp.referrer
ORDER BY tasa_cancelacion_pct DESC

**PySpark (mismo resultado, otra forma de escribirlo):**

In [0]:
w = Window.partitionBy("property_id").orderBy(F.desc("visitas"))

canal_por_propiedad = (spark.table("bigdata_grupo30.wanderbricks.clickstream_plata")
    .groupBy("property_id", "referrer")
    .agg(F.count("*").alias("visitas"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("property_id", F.col("referrer").alias("canal_predominante")))

df_bookings = spark.table("bigdata_grupo30.wanderbricks.bookings_plata")

resultado = (df_bookings.join(canal_por_propiedad, "property_id")
    .groupBy("canal_predominante")
    .agg(
        F.count("*").alias("total_reservas"),
        F.count(F.when(F.col("status") == "confirmed", 1)).alias("confirmadas"),
        F.count(F.when(F.col("status") == "cancelled", 1)).alias("canceladas"))
    .withColumn("tasa_cancelacion_pct", F.round(F.col("canceladas") / F.col("total_reservas") * 100, 2))
    .orderBy(F.desc("tasa_cancelacion_pct")))
resultado.display()

**Interpretación**

La tasa de cancelación por canal es una métrica clave porque una reserva confirmada no es sinónimo de ingreso asegurado: cada cancelación representa capacidad de la propiedad bloqueada sin generar pago, tiempo de gestión del anfitrión y, en algunos casos, reembolsos asociados. Conocer si algunos canales traen reservas más "volátiles" que otros permite a marketing anticipar qué tan sólido es realmente el tráfico que atrae cada canal, más allá de si convierte o no en el momento de reservar.

---
## 6. Conclusiones

Nuestra pregunta central de negocio buscaba entender qué combinaciones de dispositivo y canal de origen generan más reservas confirmadas y pagadas, frente a las que solo generan tráfico sin conversión. Los resultados muestran que, en realidad, ningún dispositivo ni canal domina de forma contundente: las doce combinaciones de dispositivo y canal se mueven en un rango de conversión muy estrecho, entre el 10.79% y el 11.72%, con mobile + google a la cabeza. Esto nos llevó a replantear nuestra hipótesis inicial: esperábamos encontrar un canal claramente superior, pero el dispositivo y el canal de origen, por sí solos, no parecen ser un factor determinante de conversión en Wanderbricks.

Al cruzar los resultados entre consultas encontramos hallazgos que no anticipábamos. Primero, el volumen de abandono más alto (tablet + ad, con 4,605 usuarios sin reserva) no corresponde a un canal de mala conversión, sino a uno de los que más tráfico total recibe  un canal puede parecer problemático en cifras absolutas y no serlo en proporción. Segundo, y quizás el hallazgo más relevante del trabajo: conversión, satisfacción e ingresos no se comportan igual entre canales. Google convierte bien pero deja las reseñas más bajas (2.86 sobre los cuatro canales evaluados); ad genera más ingresos brutos pero también la mayor tasa de reembolso; y email, sin ser el que más convierte ni el que más factura, resulta ser el canal más estable en todas las métricas de calidad: menor tasa de reembolso y menor tasa de cancelación (20.78%, frente a un 21.68% de direct). Esto nos permite responder con matices nuestras tres preguntas de negocio: no existe un único canal "ganador"; la decisión depende de si el objetivo es maximizar conversión, ingresos o retención a largo plazo.

Si tuviéramos que repetir este análisis, cambiaríamos dos cosas. Primero, incorporaríamos la duración de la estadía y la antelación de la reserva como variables adicionales, ya que podrían explicar mejor la conversión que el dispositivo o el canal por sí solos. Segundo, tendríamos en cuenta la limitación que encontramos en la Sección 2: los datos de clickstream no están distribuidos naturalmente en el tiempo, sino concentrados en una ventana de apenas segundos, mientras que bookings cubre casi tres años. Esto significa que nuestras conclusiones sobre navegación son válidas como una fotografía agregada del comportamiento, pero no nos permiten afirmar nada sobre tendencias o cambios de comportamiento a lo largo del tiempo.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| **Hanna Jineth Contreras Salinas** | Sección 1 (contexto y problema), Sección 2 (descripción de los datos y periodo cubierto), Sección 4.1 (catálogo, esquema y volumen), Sección 4.2 (capa bronce), Sección 4.5 (las 4 garantías de Delta: ACID, historial, time travel, evolución de esquema), y la organización y ajuste final de todo el documento. | Explica por qué se construyó la capa bronce con un bloque reutilizable para las 4 tablas, y demuestra las 4 garantías de Delta con el ejemplo del ajuste del 19% y el time travel. |
| **Arley Steven Franco Galeano** | Sección 4.3 (capa plata) y Sección 4.4 (datos semiestructurados). | Explica cómo se limpiaron las 4 tablas y cómo se aplanó el struct de `clickstream` para dejar `device` y `referrer` como columnas normales. |
| **Andres Felipe Grisales Vahos** | Sección 5 (las 5 consultas analíticas, en SQL y PySpark). | Muestra una de sus consultas ejecutándose y explica qué decisión de negocio permite tomar cada resultado. |
| **Los tres** | Sección 3 (comparación de paradigmas y justificación del modelo) y Sección 6 (conclusiones), construidas en conjunto una vez tuvimos los resultados de las 5 consultas. | Cada uno responde las 3 preguntas obligatorias desde la parte que trabajó. |

**Uso de asistentes de IA:** usamos NotebookLM (Google) como apoyo para el estudio, subiendo los laboratorios y notebooks del curso para generar mapas conceptuales que nos ayudaron a desglosar y entender mejor los conceptos de Delta Lake y Spark antes de aplicarlos. El código y las decisiones de diseño fueron discutidos y entendidos por el grupo antes de incluirlos; cada integrante puede explicar cualquier línea de la parte que le correspondió.